# Adult Income: Classical ML and PyTorch

A post-coursework extension with validation-only selection, a separate official UCI test, and three DL seeds. Original coursework is preserved. This notebook uses the modular source in `src/adult_benchmark.py`.

Install dependencies with `%pip install -r requirements-improved.txt` if needed. CPU is sufficient.

In [ ]:
from pathlib import Path
import sys
import json
import subprocess
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/adult_benchmark.py').exists()), None)
if ROOT is None:
    ROOT = Path('/content/adult-income-improvement')
    subprocess.run(['git', 'clone', 'https://github.com/VietAnh1803/HCMUT_ML_AdultCensusIncome_Pipeline.git', str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / 'src'))
OUT = ROOT / 'reports/improved'
print('Source:', ROOT)

## Run or inspect a recorded experiment
The shipped results are from the actual CPU experiment. Reuse is explicit; turn it off to download the official data and train again. Training and threshold selection finish before final test labels are parsed.

In [ ]:
REUSE_RECORDED_RUN = True
if not REUSE_RECORDED_RUN or not (OUT / 'metrics.json').exists():
    from adult_benchmark import main
    main(OUT, ROOT / '.cache/adult')
else:
    print('Reusing the recorded, verified CPU run; this cell does not retrain.')
report = json.loads((OUT / 'metrics.json').read_text())
display(pd.Series(report['split']).to_frame('Split evidence'))

## Validation comparison
Select on AP. The two legacy recipes are predefined controls. No test score is used here.

In [ ]:
validation = pd.json_normalize(report['validation_experiments'])
display(validation[['name', 'family', 'validation.average_precision', 'validation.roc_auc', 'fit_seconds']].sort_values('validation.average_precision', ascending=False))
display(pd.Series(report['selection']).to_frame('Frozen choices'))

## Final test and DL stability
Only selected models and controls are graded. These are distinct from historical coursework scores.

In [ ]:
display(pd.DataFrame(report['test']).T[['accuracy', 'average_precision', 'roc_auc', 'f1_positive', 'f1_macro', 'log_loss']])
display(pd.DataFrame(report['selected_mlp_seed_repeats'])[['seed', 'accuracy', 'average_precision', 'roc_auc']])
assert all(report['checks'].values())
print('ADULT_REPORT_VERIFIED: all recorded protocol checks passed')

## Measured comparison
Accuracy gains include validation threshold changes. AP gains are modest, and the MLP does not outperform LightGBM. See the improvement report and paired bootstrap for uncertainty. This 1994 census dataset is a historical learning benchmark.

In [ ]:
display(Image(filename=str(OUT / 'comparison.png')))
display(pd.Series(report['environment']).to_frame('Recorded environment'))